%md
## Adding a widget to the notebook for specific date

In [0]:
dbutils.widgets.text("ingestion_date", "", "Ingestion Date (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

if ingestion_date == "":
    from datetime import date
    ingestion_date = str(date.today())

BASE_PATH = "abfss://raw@stmavaluationplatform.dfs.core.windows.net"

sources = {
    "ma_multiples": f"{BASE_PATH}/ma_multiples/ingestion_date={ingestion_date}/multiples.json",
    "ma_multiples_by_year": f"{BASE_PATH}/ma_multiples/ingestion_date={ingestion_date}/multiples-by-year.json"
}

raw_dataframes = {}
for name, path in sources.items():
    raw_dataframes[name] = spark.read.option("multiline", "true").json(path)
    print(f"{name}: {raw_dataframes[name].count()} rows loaded")

ma_multiples: 1 rows loaded
ma_multiples_by_year: 1 rows loaded


#### Flattening data 

In [0]:
row = raw_dataframes["ma_multiples"].collect()[0]
multiples_dict = row.asDict(recursive=True)

records = []
for sub_vertical, brackets in multiples_dict["data"].items():
    if sub_vertical == "other" or sub_vertical.startswith("other-"):
        continue
    for ev_bracket, metrics in brackets.items():
        for metric_type, stats in metrics.items():
            records.append({
                "sub_vertical": sub_vertical,
                "ev_bracket": ev_bracket,
                "metric_type": metric_type,
                "p25": stats.get("p25"),
                "median_multiple": stats.get("p50"),
                "p75": stats.get("p75"),
                "deal_count": stats.get("n"),
                "source_year": None,
                "source_file": "ma_multiples"
            })

print(f"Flattened {len(records)} rows from ma_multiples")

Flattened 208 rows from ma_multiples


In [0]:
row = raw_dataframes["ma_multiples_by_year"].collect()[0]
by_year_dict = row.asDict(recursive=True)

records_by_year = []
for sub_vertical, years in by_year_dict["data"].items():
    if sub_vertical == "other" or sub_vertical.startswith("other-"):
        continue
    for year, year_data in years.items():
        for metric_type in ("ev_ebitda", "ev_revenue"):
            if metric_type in year_data:
                stats = year_data[metric_type]
                records_by_year.append({
                    "sub_vertical": sub_vertical,
                    "ev_bracket": None,
                    "metric_type": metric_type,
                    "p25": stats.get("p25"),
                    "median_multiple": stats.get("p50"),
                    "p75": stats.get("p75"),
                    "deal_count": stats.get("n"),
                    "source_year": int(year),
                    "source_file": "ma_multiples_by_year"
                })

print(f"Flattened {len(records_by_year)} rows from ma_multiples_by_year")

Flattened 196 rows from ma_multiples_by_year


### Fact Table — silver_valuation_multiple

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, lit, col
from pyspark.sql.types import DoubleType, IntegerType

all_records = records + records_by_year

silver_df = spark.createDataFrame(all_records) \
    .withColumn("p25", col("p25").cast(DoubleType())) \
    .withColumn("median_multiple", col("median_multiple").cast(DoubleType())) \
    .withColumn("p75", col("p75").cast(DoubleType())) \
    .withColumn("deal_count", col("deal_count").cast(IntegerType())) \
    .withColumn("source_year", col("source_year").cast(IntegerType())) \
    .withColumn("valuation_id", monotonically_increasing_id()) \
    .withColumn("ingestion_date", lit(ingestion_date))

display(silver_df.limit(10))
print(f"Total rows: {silver_df.count()}")

deal_count,ev_bracket,median_multiple,metric_type,p25,p75,source_file,source_year,sub_vertical,valuation_id,ingestion_date
11,25m_100m_ev,1.75,ev_revenue,0.85,4.66,ma_multiples,null,advertising-agency,0,2026-08-18
14,5m_25m_ev,1.29,ev_revenue,0.85,1.67,ma_multiples,null,advertising-agency,1,2026-08-18
24,100m_500m_ev,2.38,ev_revenue,1.35,3.06,ma_multiples,null,aerospace,2,2026-08-18
13,25m_100m_ev,1.86,ev_revenue,0.76,2.93,ma_multiples,null,aerospace,3,2026-08-18
10,5m_25m_ev,0.77,ev_revenue,0.61,1.02,ma_multiples,null,aerospace,4,2026-08-18
24,over_500m_ev,14.25,ev_ebitda,12.9,16.92,ma_multiples,null,aerospace,5,2026-08-18
32,over_500m_ev,2.12,ev_revenue,1.67,3.29,ma_multiples,null,aerospace,6,2026-08-18
11,25m_100m_ev,9.5,ev_ebitda,6.8,12.2,ma_multiples,null,ambulatory-surgery-center,7,2026-08-18
13,25m_100m_ev,2.5,ev_revenue,1.95,3.2,ma_multiples,null,ambulatory-surgery-center,8,2026-08-18
20,5m_25m_ev,8.1,ev_ebitda,5.47,11.38,ma_multiples,null,ambulatory-surgery-center,9,2026-08-18


Total rows: 404


In [0]:
silver_staging_path = "abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_valuation_multiple_staging"
silver_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver_staging_path)
print(f"Written {silver_df.count()} rows to staging: {silver_staging_path}")

Written 404 rows to staging: abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_valuation_multiple_staging


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver_valuation_multiple_staging
USING DELTA
LOCATION '{silver_staging_path}'
""")

display(spark.sql("SELECT COUNT(*) as row_count FROM silver_valuation_multiple_staging"))

row_count
404


### Build Dimension — silver_ev_bracket

In [0]:
ev_bracket_data = [
    ("under_5m_ev", "<$5M", 0, 5_000_000, 1),
    ("5m_25m_ev", "$5M-$25M", 5_000_000, 25_000_000, 2),
    ("25m_100m_ev", "$25M-$100M", 25_000_000, 100_000_000, 3),
    ("100m_500m_ev", "$100M-$500M", 100_000_000, 500_000_000, 4),
    ("over_500m_ev", ">$500M", 500_000_000, None, 5),
]

silver_ev_bracket_df = spark.createDataFrame(
    ev_bracket_data,
    ["ev_bracket", "ev_bracket_label", "min_ev", "max_ev", "bracket_order"]
).withColumn("ev_bracket_id", monotonically_increasing_id())

actual_brackets = set(r[0] for r in silver_df.select("ev_bracket").distinct().collect() if r[0] is not None)
expected_brackets = set(row[0] for row in ev_bracket_data)
print("Match:", actual_brackets == expected_brackets, "| Actual:", actual_brackets)

display(silver_ev_bracket_df)

Match: True | Actual: {'25m_100m_ev', '5m_25m_ev', '100m_500m_ev', 'under_5m_ev', 'over_500m_ev'}


ev_bracket,ev_bracket_label,min_ev,max_ev,bracket_order,ev_bracket_id
under_5m_ev,<$5M,0,5000000,1,0
5m_25m_ev,$5M-$25M,5000000,25000000,2,1
25m_100m_ev,$25M-$100M,25000000,100000000,3,2
100m_500m_ev,$100M-$500M,100000000,500000000,4,3
over_500m_ev,>$500M,500000000,null,5,4


In [0]:
ev_bracket_path = "abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_ev_bracket"
silver_ev_bracket_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ev_bracket_path)
print(f"Written {silver_ev_bracket_df.count()} rows to {ev_bracket_path}")

Written 5 rows to abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_ev_bracket


### Build Dimension — silver_industry

In [0]:
industry_group_rules = [
    ("Technology", ["saas", "software", "digital-media", "gaming", "ecommerce", "-it", "healthtech", "it-services"]),
    ("Healthcare", ["health", "medical", "dental", "pharmacy", "therapy", "veterinary", "mental",
                     "surgery", "hospice", "laboratory"]),
    ("Energy", ["oil-gas", "electrical-utility", "energy"]),
    ("Industrial", ["aerospace", "industrial", "metal-fabrication", "packaging", "plastics",
                     "auto-parts", "specialty-contractor", "freight", "trucking",
                     "wholesale-distribution", "electronics"]),
    ("Consumer", ["apparel", "consumer-products", "food", "beverage", "restaurant",
                   "specialty-retail", "auto-dealership"]),
    ("Professional Services", ["consulting", "advertising-agency", "radio-television"]),
    ("Financial", ["financial", "bank", "insurance"]),
]

def classify_industry(sub_vertical):
    for group, keywords in industry_group_rules:
        if any(kw in sub_vertical for kw in keywords):
            return group
    return "Other"

distinct_sub_verticals = [r[0] for r in silver_df.select("sub_vertical").distinct().collect()]

industry_records = [
    {"sub_vertical": sv, "industry_group": classify_industry(sv), "industry_status": "Active"}
    for sv in distinct_sub_verticals
]

silver_industry_df = spark.createDataFrame(industry_records) \
    .withColumn("industry_id", monotonically_increasing_id())

display(silver_industry_df.orderBy("industry_group", "sub_vertical"))
print(f"Total sub-verticals: {len(distinct_sub_verticals)}")

industry_group,industry_status,sub_vertical,industry_id
Consumer,Active,apparel,9
Consumer,Active,auto-dealership,10
Consumer,Active,beverage-manufacturing,22
Consumer,Active,consumer-products,16
Consumer,Active,food-distribution,12
Consumer,Active,food-manufacturing,23
Consumer,Active,restaurant-qsr,45
Consumer,Active,specialty-retail,42
Energy,Active,electrical-utility,15
Energy,Active,oil-gas-services,33


Total sub-verticals: 46


In [0]:
industry_path = "abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_industry"
silver_industry_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(industry_path)
print(f"Written {silver_industry_df.count()} rows to {industry_path}")

Written 46 rows to abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_industry


### catalog tables

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_ev_bracket
USING DELTA
LOCATION 'abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_ev_bracket'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver_industry
USING DELTA
LOCATION 'abfss://silver@stmavaluationplatform.dfs.core.windows.net/silver_industry'
""")

display(spark.sql("SHOW TABLES"))

database,tableName,isTemporary
default,gold_data_quality,false
default,silver_ev_bracket,false
default,silver_industry,false
default,silver_valuation_multiple,false
default,silver_valuation_multiple_staging,false
default,silver_valuation_multiple_validated,false
